#  1. Business Context
Le client SportsStats souhaite produire des analyses permettant aux journalistes sportifs et aux entraîneurs d'identifier les tendances marquantes des Jeux Olympiques à partir de plus de 120 ans de données.

Toutes les analyses de ce notebook utiliseront la vue :**vw_olympics_analytics**

##  Country Performance Analysis

Business Questions

1. Which countries have won the highest number of Olympic medals throughout Olympic history?
2. What is the distribution of Gold, Silver, and Bronze medals for each country?
3. Which countries have the highest medal rate based on the number of athletes they sent to the Olympic Games?

In [0]:
-- Top 10 countries with the highest number of Olympic medals
SELECT COUNT(*) AS Total_Medals,
    Region
FROM vw_olympics_analytics
WHERE Is_Medalist = 1
GROUP BY Region
ORDER BY Total_Medals DESC
LIMIT 10;

Total_Medals,Region
5637,USA
3947,Russia
3756,Germany
2068,UK
1777,France
1637,Italy
1536,Sweden
1352,Canada
1349,Australia
1135,Hungary


In [0]:
-- What is the distribution of Gold, Silver, and Bronze medals for each country?
WITH country_medal_summary AS(
  SELECT Region,
    SUM(CASE WHEN Medal = 'Gold' THEN 1 ELSE 0 END) AS Gold_Medals,
    SUM(CASE WHEN Medal = 'Silver' THEN 1 ELSE 0 END) AS Silver_Medals,
    SUM(CASE WHEN Medal = 'Bronze' THEN 1 ELSE 0 END) AS Bronze_Medals,
    COUNT(*) AS Total_Medals
    
  FROM vw_olympics_analytics
  WHERE Is_Medalist = 1
  GROUP BY Region
)
SELECT Region,
   Gold_Medals,
   Silver_Medals,
   Bronze_Medals,
   Total_Medals
FROM country_medal_summary
ORDER BY Gold_Medals DESC, Total_Medals DESC
LIMIT 10;


Region,Gold_Medals,Silver_Medals,Bronze_Medals,Total_Medals
USA,2638,1641,1358,5637
Russia,1599,1170,1178,3947
Germany,1301,1195,1260,3756
UK,678,739,651,2068
Italy,575,531,531,1637
France,501,610,666,1777
Sweden,479,522,535,1536
Canada,463,438,451,1352
Hungary,432,332,371,1135
Norway,378,361,294,1033


In [0]:
--Which countries have the highest medal rate based on the number of athletes they sent to the Olympic Games?
WITH country_medal_summary AS(
  SELECT Region,
    
    COUNT(*) AS Total_Medals
    
  FROM vw_olympics_analytics
  WHERE Is_Medalist = 1
  GROUP BY Region
),
  total_athlete_by_country AS(

    SELECT Region,
          COUNT(DISTINCT ID) AS Total_Athlete
        FROM vw_olympics_analytics
        GROUP BY Region

        
  )
SELECT c.Region,
   t.Total_Athlete,
   c.Total_Medals,
    round(c.Total_Medals *1.0/nullif(t.Total_Athlete,0),3)  AS Medals_Per_Athlete
   FROM country_medal_summary c
   LEFT JOIN total_athlete_by_country t 
   ON c.Region = t.Region 
ORDER BY Medals_Per_Athlete DESC, Total_Medals DESC
LIMIT 10;


Region,Total_Athlete,Total_Medals,Medals_Per_Athlete
Russia,5610,3947,0.704
USA,9653,5637,0.584
Germany,7575,3756,0.496
Norway,2216,1033,0.466
Jamaica,370,157,0.424
Hungary,2761,1135,0.411
Sweden,3787,1536,0.406
Finland,2347,900,0.383
Romania,1798,653,0.363
Netherlands,2939,1040,0.354


### Results Interpretation
1. Les États-Unis dominent le classement historique des médailles olympiques, en tête à la fois pour le nombre total de médailles et pour les médailles d'or. Plusieurs pays européens montrent aussi des performances constamment solides dans toutes les catégories de médailles. Les pays avec un grand nombre de médailles d'or se classent généralement aussi haut dans le total des médailles, ce qui indique une excellence compétitive durable.

2.De petites délégations comme le Kenya peuvent obtenir une meilleure efficacité en médailles que des pays envoyant des milliers d'athlètes. Les grandes équipes olympiques n'ont pas forcément le meilleur ratio médailles-par-athlète. 

###  Business Value
1. Les journalistes sportifs peuvent rapidement repérer les nations qui ont dominé historiquement avant les grands événements olympiques. Les entraîneurs et les analystes peuvent comparer les performances nationales avec celles des principaux pays olympiques. Les résultats offrent une base fiable pour les classements Power BI et l'analyse des tendances historiques.
2. SportsStats peut comparer l'efficacité des délégations olympiques indépendamment de la taille des pays. Les journalistes peuvent mettre en avant les nations performantes qui maximisent les performances de leurs athlètes.Et les entraîneurs et analystes peuvent suivre l'efficacité olympique nationale au fil du temps.

## 4. Athlete Performance  and Sports Analysis
1. Which athletes are the most decorated in Olympic history?
2. Which sports have the highest medal concentration, and how does this vary by gender?

In [0]:
-- Which athletes are the most decorated in Olympic history?
SELECT ID, 
   Name,
   Region,
   SUM(CASE WHEN Medal = 'Gold' THEN 1 ELSE 0 END) AS Gold_Medals,
   SUM(CASE WHEN Medal = 'Silver' THEN 1 ELSE 0 END) AS Silver_Medals,
   SUM(CASE WHEN Medal = 'Bronze' THEN 1 ELSE 0 END) AS Bronze_Medals,
   SUM (Is_Medalist) AS Athlete_Total_Medals
FROM vw_olympics_analytics
GROUP BY ID, Name, Region
ORDER BY Athlete_Total_Medals DESC
LIMIT 10;

    

ID,Name,Region,Gold_Medals,Silver_Medals,Bronze_Medals,Athlete_Total_Medals
94406,"Michael Fred Phelps, II",USA,23,3,2,28
67046,Larysa Semenivna Latynina (Diriy-),Russia,9,5,4,18
4198,Nikolay Yefimovich Andrianov,Russia,7,5,3,15
11951,Ole Einar Bjrndalen,Norway,8,4,1,13
89187,Takashi Ono,Japan,5,4,4,13
74420,Edoardo Mangiarotti,Italy,6,5,2,13
109161,Borys Anfiyanovych Shakhlin,Russia,7,4,2,13
35550,Birgit Fischer-Schmidt,Germany,8,4,0,12
87390,Paavo Johannes Nurmi,Finland,9,3,0,12
85286,Aleksey Yuryevich Nemov,Russia,4,2,6,12


Databricks visualization. Run in Databricks to view.

In [0]:
--Which sports have the highest medal concentration, and how does this vary by gender?
SELECT Sport,
   sum (CASE WHEN Sex = 'M' THEN 1 ELSE 0 END )AS Male_Medals,
   sum (CASE WHEN Sex = 'F' THEN 1 ELSE 0 END) AS Female_Medals,
   sum (Is_Medalist) AS Sport_Total_Medals
   FROM vw_olympics_analytics
   WHERE Is_Medalist = 1
   GROUP BY Sport
   ORDER BY Sport_Total_Medals DESC
   LIMIT 10;


Sport,Masculin_Medals,Feminin_Medals,Sport_Total_Medals
Athletics,2694,1275,3969
Swimming,1674,1374,3048
Rowing,2225,720,2945
Gymnastics,1555,701,2256
Fencing,1394,349,1743
Football,1269,302,1571
Ice Hockey,1230,300,1530
Hockey,1050,478,1528
Wrestling,1228,68,1296
Cycling,1087,176,1263


Databricks visualization. Run in Databricks to view.

### Results Interpretation
1. L'analyse met en avant les athlètes qui ont accumulé le plus grand nombre de médailles olympiques au fil de l'histoire. En tête on a l'athlete américain Michael Fred Phelps, II avec un total de 28 médailles dont 23 en or, 3 en argent et 2 en bronze
2. L'athlétisme et la natation représentent le plus grand nombre de médailles olympiques.  
Certains sports restent majoritairement masculins, tandis que d'autres montrent une répartition des sexes plus équilibrée. La participation selon le genre varie considérablement selon le sport.

### Business Value
1. Les journalistes sportifs peuvent rapidement identifier les athlètes olympiques légendaires pour des comparaisons historiques et la couverture des événements. Les entraîneurs et les organisations sportives peuvent aussi analyser l'excellence sportive à long terme dans différents sports et pays.
2. SportsStats peut identifier les disciplines avec la plus grande profondeur compétitive. Les journalistes peuvent comparer la représentation des genres dans les sports olympiques. Les fédérations nationales peuvent évaluer les tendances de participation selon le genre sur le long terme.

## 6. Olympic Trends
1. How has Olympic participation and medal distribution evolved over time?

In [0]:
-- Medals Evolution
SELECT
    Year,
    SUM(Is_Medalist) AS Total_Medals
FROM vw_olympics_analytics
GROUP BY Year
ORDER BY Year;

Year,Total_Medals
1896,143
1900,604
1904,486
1906,458
1908,831
1912,941
1920,1308
1924,962
1928,823
1932,739


Databricks visualization. Run in Databricks to view.

In [0]:
-- Participation Evolution
SELECT Year,
    count(DISTINCT ID) AS Total_Athlete
FROM vw_olympics_analytics
GROUP BY  Year
ORDER BY Year 
    

Year,Total_Athlete
1896,176
1900,1224
1904,650
1906,841
1908,2024
1912,2409
1920,2676
1924,3565
1928,3703
1932,2174


Databricks visualization. Run in Databricks to view.

### Results Interpretation

La participation aux Jeux olympiques a beaucoup augmenté au cours du dernier siècle. Le nombre d'athlètes participants a grandi en même temps que l'expansion des épreuves olympiques. La production de médailles a suivi une tendance similaire à la hausse, reflétant l'ampleur croissante des Jeux.

### Business Value

SportsStats peut identifier les tendances de croissance olympique à long terme. Les journalistes peuvent montrer comment les Jeux ont évolué à travers différentes époques. Les organisations sportives peuvent comparer la participation historique avec les éditions modernes des Jeux Olympiques.

## 7. Executive Summary

1. Les États-Unis dominent le classement historique des médailles. 
2. Les délégations plus petites comme le Kenya obtiennent une efficacité médaille plus élevée.
3. Michael Phelps est l'olympien le plus décoré.
4. L'athlétisme et la natation concentrent le plus grand nombre de médailles.
5. La participation aux Jeux olympiques a augmenté régulièrement au cours du siècle dernier.